# V6.7 — Finding Destiny ablation evaluation on Finding Nemo audit V5

Evaluate interchangeable YOLO26m ablation checkpoints from both Finding Destiny and Finding Dori on the 1,931-image `test_main_v5_full_consensus_assisted` dataset. Results and plots are saved locally and mirrored to Camber.

This notebook does not run evaluation until the **Run selected evaluations** cell is executed.

## 1. Paths and editable checkpoint registry

In [1]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, time
import yaml

def find_repo(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'benchmakring_transmission_line_defect_model').is_dir():
            return candidate
    raise FileNotFoundError('Repository root not found')

REPO = find_repo(Path.cwd())
BENCHMARK = REPO / 'benchmakring_transmission_line_defect_model'
TEST_ROOT = REPO / 'data/working/finding_nemo_test_audit_v5_bundle/data/test_main_v5_full_consensus_assisted'
TEST_YAML = TEST_ROOT / 'dataset.yaml'
DESTINY_WEIGHTS_ROOT = BENCHMARK / 'weights/finding_destiny'
DORI_WEIGHTS_ROOT = BENCHMARK / 'weights/finding_dori'
RESULTS_ROOT = BENCHMARK / 'results/finding_destiny_test_audit_v5'
REMOTE_ROOT = 'stash://bunpmc/projects/finding_destiny/test_audit_v5_results'

# Edit this dictionary to replace a checkpoint or add another ablation.
WEIGHTS = {
    'destiny_m124': DESTINY_WEIGHTS_ROOT / 'yolo26m_m_124/weights/best.pt',
    'destiny_m134': DESTINY_WEIGHTS_ROOT / 'yolo26m_m_134/weights/best.pt',
    'destiny_m234': DESTINY_WEIGHTS_ROOT / 'yolo26m_m_234/weights/best.pt',
    'dori_m1_abf': DORI_WEIGHTS_ROOT / 'yolo26m_m1_a_b_f/weights/best.pt',
    'dori_m2_cd': DORI_WEIGHTS_ROOT / 'yolo26m_m2_c_d/weights/best.pt',
    'dori_m3_eg': DORI_WEIGHTS_ROOT / 'yolo26m_m3_e_g/weights/best.pt',
}
# Evaluate any subset, e.g. ['m_124'], or all registered checkpoints.
SELECTED = list(WEIGHTS)
IMGSZ, BATCH, DEVICE, WORKERS = 640, 8, 0, 2
print('Test YAML:', TEST_YAML)
print('Local results:', RESULTS_ROOT)
print('Camber results:', REMOTE_ROOT)
print('Selected:', SELECTED)

Test YAML: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\data\working\finding_nemo_test_audit_v5_bundle\data\test_main_v5_full_consensus_assisted\dataset.yaml
Local results: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\results\finding_destiny_test_audit_v5
Camber results: stash://bunpmc/projects/finding_destiny/test_audit_v5_results
Selected: ['destiny_m124', 'destiny_m134', 'destiny_m234', 'dori_m1_abf', 'dori_m2_cd', 'dori_m3_eg']


## 2. Read-only preflight

In [2]:
cfg = yaml.safe_load(TEST_YAML.read_text(encoding='utf-8'))
freeze = json.loads((TEST_ROOT / 'freeze.json').read_text(encoding='utf-8'))
assert Path(cfg['path']).resolve() == TEST_ROOT.resolve(), (cfg['path'], TEST_ROOT)
assert cfg['test'] == 'images/test' and len(cfg['names']) == 6
images = [p for p in (TEST_ROOT/'images/test').iterdir() if p.is_file() and p.suffix != '.cache']
labels = list((TEST_ROOT/'labels/test').glob('*.txt'))
# V5 freeze.json records consensus provenance, not an image-count field.
assert len(images) == len(labels) == 1931, (len(images), len(labels))
assert freeze['name'] == 'test_main_v5_full_consensus_assisted', freeze.get('name')
assert SELECTED and not (set(SELECTED) - set(WEIGHTS)), SELECTED
for name in SELECTED:
    path = Path(WEIGHTS[name])
    assert path.is_file() and path.stat().st_size > 40_000_000, (name, path)
    print(f'{name:8} {path.stat().st_size/1024**2:6.1f} MiB  {path}')
print(f'Finding Nemo audit V5: {len(images)} images, {len(labels)} labels')

destiny_m124   42.0 MiB  D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\weights\finding_destiny\yolo26m_m_124\weights\best.pt
destiny_m134   42.0 MiB  D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\weights\finding_destiny\yolo26m_m_134\weights\best.pt
destiny_m234   42.0 MiB  D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\weights\finding_destiny\yolo26m_m_234\weights\best.pt
dori_m1_abf   42.0 MiB  D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\weights\finding_dori\yolo26m_m1_a_b_f\weights\best.pt
dori_m2_cd   42.0 MiB  D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\weights\finding_dori\yolo26m_m2_c_d\weights\best.pt


## 3. Camber upload and metric helpers

In [3]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def load_camber_key():
    if os.environ.get('CAMBER_API_KEY'):
        return
    for line in (REPO/'.env').read_text(encoding='utf-8').splitlines():
        key, sep, value = line.partition('=')
        if sep and key.strip() == 'CAMBER_API_KEY' and value.strip():
            os.environ['CAMBER_API_KEY'] = value.strip().strip(chr(34)).strip(chr(39))
            return
    raise RuntimeError('CAMBER_API_KEY missing from repository .env')

def wsl_path(path: Path) -> str:
    path = path.resolve()
    return f'/mnt/{path.drive.rstrip(chr(58)).lower()}{path.as_posix().split(chr(58),1)[1]}'

def camber_cp(local: Path, remote: str, attempts=3):
    load_camber_key()
    env = os.environ.copy()
    parts = [part for part in env.get('WSLENV','').split(':') if part]
    if 'CAMBER_API_KEY' not in parts:
        parts.append('CAMBER_API_KEY')
    env['WSLENV'] = ':'.join(parts)
    cmd = ['wsl','--','/home/dotru/.camber/bin/camber','stash','cp',wsl_path(local),remote]
    for attempt in range(1, attempts+1):
        proc = subprocess.run(cmd,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
        if proc.returncode == 0:
            return
        if attempt < attempts:
            time.sleep(5*attempt)
    raise RuntimeError(proc.stdout[-2000:])

def upload_folder(local_dir: Path, remote_dir: str):
    for path in sorted(local_dir.rglob('*')):
        if path.is_file() and path.suffix != '.cache':
            camber_cp(path, f'{remote_dir}/{path.relative_to(local_dir).as_posix()}')

def metric_record(name, weight, metrics):
    names = metrics.names if isinstance(metrics.names, dict) else dict(enumerate(metrics.names))
    record = {
        'ablation': name, 'weight': str(weight.resolve()), 'weight_sha256': sha256(weight),
        'test_images': 1931, 'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
        'speed_ms': {k: float(v) for k,v in metrics.speed.items()},
        'per_class_map50_95': {names[i]: float(value) for i,value in enumerate(metrics.box.maps)},
    }
    return record

## 4. Evaluate one checkpoint

In [4]:
from ultralytics import YOLO

def evaluate_one(name: str):
    if name not in WEIGHTS:
        raise KeyError(name)
    weight = Path(WEIGHTS[name]).resolve()
    out_dir = RESULTS_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'[{name}] evaluating {weight.name} on Finding Nemo audit V5', flush=True)
    metrics = YOLO(str(weight)).val(
        data=str(TEST_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS,
        project=str(RESULTS_ROOT), name=name, exist_ok=True, plots=True, save_json=False)
    record = metric_record(name, weight, metrics)
    (out_dir/'metrics.json').write_text(json.dumps(record,indent=2),encoding='utf-8')
    upload_folder(out_dir, f'{REMOTE_ROOT}/{name}')
    print(f'[{name}] local={out_dir} remote={REMOTE_ROOT}/{name}', flush=True)
    return record

## 5. Run selected evaluations

Edit `SELECTED` in the first cell before running if only a subset is needed. Completed `metrics.json` files are overwritten deliberately when re-evaluated.

In [5]:
records = []
for name in SELECTED:
    records.append(evaluate_one(name))

[destiny_m124] evaluating best.pt on Finding Nemo audit V5
Ultralytics 8.4.172  Python-3.12.2 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26m summary (fused): 130 layers, 20,354,078 parameters, 0 gradients, 68.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 27.242.0 MB/s, size: 163.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\data\working\finding_nemo_test_audit_v5_bundle\data\test_main_v5_full_consensus_assisted\labels\test... 1931 images, 124 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1931/1931 694.3it/s 2.8s0.0s
val: New cache created: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\data\working\finding_nemo_test_audit_v5_bundle\data\test_main_v5_full_consensus_assisted\labels\test.cache
                 Class  

## 6. Consolidate comparison and upload summary

In [6]:
available = []
for name in WEIGHTS:
    path = RESULTS_ROOT/name/'metrics.json'
    if path.is_file():
        available.append(json.loads(path.read_text(encoding='utf-8')))
assert available, 'No metrics.json found; run at least one evaluation first'
available.sort(key=lambda row: row['map50_95'], reverse=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
summary_json = RESULTS_ROOT/'summary.json'
summary_csv = RESULTS_ROOT/'summary.csv'
summary_json.write_text(json.dumps(available,indent=2),encoding='utf-8')
fields = ['ablation','weight_sha256','test_images','precision','recall','map50','map50_95']
with summary_csv.open('w',newline='',encoding='utf-8') as handle:
    writer = csv.DictWriter(handle,fieldnames=fields); writer.writeheader()
    writer.writerows({key:row[key] for key in fields} for row in available)
camber_cp(summary_json,f'{REMOTE_ROOT}/summary.json')
camber_cp(summary_csv,f'{REMOTE_ROOT}/summary.csv')
for row in available:
    print(f"{row['ablation']:8} mAP50={row['map50']:.4f} mAP50-95={row['map50_95']:.4f}")

dori_m1_abf mAP50=0.7898 mAP50-95=0.4333
dori_m2_cd mAP50=0.7926 mAP50-95=0.4321
dori_m3_eg mAP50=0.7914 mAP50-95=0.4312
destiny_m134 mAP50=0.7935 mAP50-95=0.4231
destiny_m124 mAP50=0.7967 mAP50-95=0.4141
destiny_m234 mAP50=0.7896 mAP50-95=0.4105
